In [0]:
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.ventes.checkpoints")

In [0]:
from pyspark.sql.functions import col, current_timestamp

# on lit les données. Auto Loader surveille le dossier depot et prépare la lecture des fichiers CSV, avec deux colonnes en plus : le nom du fichier et l’heure de chargement.
df = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.schemaLocation", "/Volumes/workspace/ventes/checkpoints/schema_ventes") 
    # Auto Loader devine les colonnes et leurs types (id_vente, date, magasin…). Il enregistre ce schéma dans ce dossier pour s’en souvenir d’une exécution à l’autre.
    .load("/Volumes/workspace/ventes/depot/") # Dossier surveillé : la boîte aux lettres.
    .select(
        "*",
        col("_metadata.file_name").alias("fichier_source"),
        current_timestamp().alias("charge_le"),
        # "*" garde toutes les colonnes d’origine. _metadata.file_name est une information que Databricks ajoute automatiquement : le nom du fichier d’où vient la ligne. On le renomme fichier_source. current_timestamp() ajoute la date et l’heure du chargement, dans charge_le. 
    )
)

# on écrit les données. On envoie ce qui a été lu vers la table bronze_ventes, en notant dans le checkpoint les fichiers déjà traités.
(
    df.writeStream
    .option("checkpointLocation", "/Volumes/workspace/ventes/checkpoints/bronze_ventes")
    .trigger(availableNow=True)
    .toTable("workspace.ventes.bronze_ventes")
    .awaitTermination()
)

In [0]:
display(spark.sql("""
    SELECT fichier_source, COUNT(*) AS nb_lignes
    FROM workspace.ventes.bronze_ventes
    GROUP BY fichier_source
    ORDER BY fichier_source
"""))